# 08 — Multiprocessing and Parallelism

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('advanced/08', {1: 'e943c1a09917417f', 2: '7ba3b678b07b8c9a', 3: '18fa7b6422fb093e', 4: '9f55b37dde4e9645', 5: 'bf7d232f47a6d73b', 6: '867c248fab026f95'})

## 1. The GIL and processes

- CPython's GIL lets one thread run Python bytecode at a time, so threads don't speed up CPU-heavy Python.
- Processes each have their own interpreter: `concurrent.futures.ProcessPoolExecutor` runs CPU work in parallel.
- Data sent to a process is pickled: functions must be defined at module top level, and arguments must be picklable.

In [ ]:
import os, pickle
print("CPUs:", os.cpu_count())
print(len(pickle.dumps({"n": list(range(100))})), "bytes to send")

**Exercise 1.** CPU-heavy pure Python work: `"threads"` or `"processes"`?

In [ ]:
answer = ___
check(1, answer)

**Exercise 2.** Can a `lambda` be pickled to send to a process? Write `True` or `False`.

In [ ]:
import pickle
try:
    pickle.dumps(lambda x: x)
    ok = True
except Exception:
    ok = False
predicted = ___
check(2, predicted)

**Exercise 3.** Which executor runs work in separate processes? Write its class name.

In [ ]:
answer = ___
check(3, answer)

## 2. Main guard and chunking

- Start pools under `if __name__ == "__main__":`; on Windows and macOS child processes re-import your file.
- Each task has overhead (start-up, pickling). Send work in chunks: `pool.map(func, items, chunksize=100)`.
- Measure: parallel code is only worth it when the work outweighs the overhead.

In [ ]:
def chunks(items, size):
    return [items[i:i + size] for i in range(0, len(items), size)]

print(chunks(list(range(10)), 4))

**Exercise 4.** Split 10 items into chunks of 4. How many chunks?

In [ ]:
items = list(range(10))
chunks = [items[i:i + 4] for i in range(0, len(items), 4)]
predicted = ___
check(4, predicted)

**Exercise 5.** Make chunks of size 3 with slicing.

In [ ]:
items = list(range(7))
size = 3
chunks = [items[i:i + size] for i in range(0, len(items), ___)]
check(5, chunks)

**Exercise 6.** Which guard protects the pool start-up? Write the `__name__` value it compares to.

In [ ]:
answer = ___
check(6, answer)

## Independent build

Plan work for a process pool. Implement split_jobs(total, workers) returning one (start, stop) half-open interval per worker. Cover range(total) once, in order. Sizes differ by at most one; earlier workers get the extra jobs. total is nonnegative. Reject workers below 1 with ValueError. Explain why sending intervals can cost less than sending lists.

Adapt a copy of the companion process-pool script to sum the integers in each interval. Using one interval per job, compare the sum of the partial sums with sum(range(total)) for total=0 and total=7. Record where the worker must be defined so a spawned process can import it.

For verified mastery, also save this build as `split_jobs` in `practice/exercises/advanced_independent_builds.py`. Incorrect or incomplete builds cannot receive mastery credit.

In [ ]:
# Write your implementation here.

In [ ]:
# Independent mastery check: separate from guided learning feedback.
build_assessment = Exercises('advanced/08', {}, practice='advanced_independent_builds')
build_assessment('split_jobs')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** For CPU-bound pure-Python work, use…

   - **a)** asyncio
   - **b)** Processes
   - **c)** Threads

**2.** Data sent to a worker process must be…

   - **a)** A string
   - **b)** Global
   - **c)** Picklable

**3.** Why put ProcessPoolExecutor code under `if __name__ == '__main__':`?

   - **a)** Workers re-import the module; the guard stops them re-running it
   - **b)** Speed
   - **c)** Style only

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/08', {1: 'd604a269e1b8', 2: '1b51b24e890b', 3: '8fe1ad84a8e6'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [09 — Retry Policies, Rate Limiting, and Idempotency](09_retries_rate_limits_idempotency.ipynb)